# GFS weather processing

This notebook cleans historical GFS forecasts, performs quality control,
reduces the spatial grid to representative Spanish cities, and produces
hourly D+1 weather features for the electricity-demand forecasting pipeline.

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

from energy_forecasting.weather import (
    load_weather_history,
    mask_temperature_range,
    temporal_consistency_audit,
    )

In [2]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

weather_dir = project_root / "data" / "raw" / "weather"

weather_dataset = load_weather_history(weather_dir)

weather_dataset = weather_dataset.drop_vars(
    "lead_available",
    errors="ignore",
)

## Physical-range screening

Some raw GFS fields contain invalid fill values that are not decoded as NaN.
We apply a deliberately broad physical plausibility range of 150–350 K.
This is a data-quality filter, not climatological clipping.

In [3]:
temperature_clean, invalid_temperature = mask_temperature_range(
    weather_dataset["temperature_K"],
    min_k=150,
    max_k=350,
)

weather_dataset["temperature_K"] = temperature_clean

print(
    "Invalid temperature cells:",
    int(invalid_temperature.sum())
)

Invalid temperature cells: 3100


## Temporal-consistency screening

Physical bounds cannot detect every corrupt field because some erroneous
values remain meteorologically plausible. We therefore compare each interior
3-hour forecast lead with the linear midpoint of its two neighbouring leads.

The edge leads f021 and f048 are not evaluated by this symmetric diagnostic.

In [4]:
audit = temporal_consistency_audit(
    weather_dataset["temperature_K"]
)

audit_df = audit.to_dataframe()

In [5]:
audit_df.sort_values(
    "max_residual",
    ascending=False,
).head(10)

max_residual  fraction_gt_10K  fraction_gt_20K
run_time_utc forecast_hour                                                
2024-12-02   30                37.879608         0.176880         0.016080
             33                20.665009         0.017818         0.001738
             27                19.718750         0.016949         0.000000
2023-07-28   42                16.555969         0.001304         0.000000
2017-01-19   42                15.949982         0.001738         0.000000
2025-06-28   39                14.840210         0.004346         0.000000
             36                14.237061         0.004781         0.000000
2024-07-28   39                14.022247         0.016949         0.000000
             36                13.721741         0.014776         0.000000
2026-06-24   36                13.506226         0.003477         0.000000

In [6]:
audit_df.sort_values(
    "fraction_gt_10K",
    ascending=False,
).head(10)

max_residual  fraction_gt_10K  fraction_gt_20K
run_time_utc forecast_hour                                                
2024-12-02   30                37.879608         0.176880         0.016080
             33                20.665009         0.017818         0.001738
2024-07-28   39                14.022247         0.016949         0.000000
2024-12-02   27                19.718750         0.016949         0.000000
2024-07-28   36                13.721741         0.014776         0.000000
2022-06-17   39                12.263977         0.009996         0.000000
2022-08-12   36                11.703583         0.009996         0.000000
2025-08-11   39                12.651947         0.007823         0.000000
2026-07-09   39                12.021942         0.007388         0.000000
2025-08-12   36                12.506836         0.006953         0.000000

In [7]:
bad_leads = (
    (audit["max_residual"] > 25)
    | (audit["fraction_gt_10K"] > 0.05)
)

audit.where(bad_leads, drop=True)

<xarray.Dataset> Size: 32B
Dimensions:          (run_time_utc: 1, forecast_hour: 1)
Coordinates:
  * run_time_utc     (run_time_utc) datetime64[ns] 8B 2024-12-02
  * forecast_hour    (forecast_hour) int32 4B 30
Data variables:
    max_residual     (run_time_utc, forecast_hour) float32 4B 37.88
    fraction_gt_10K  (run_time_utc, forecast_hour) float64 8B 0.1769
    fraction_gt_20K  (run_time_utc, forecast_hour) float64 8B 0.01608

In [8]:
weather_dataset["temperature_K"] = (
    weather_dataset["temperature_K"]
    .where(~bad_leads)
)

## Missing-data assessment and interpolation

After physical and temporal quality screening, some forecast leads are fully
or partially missing. We quantify both cases before repairing them.

Missing temperatures are then linearly interpolated along forecast lead time.
Because GFS forecasts are available every 3 hours, this uses neighbouring
forecast leads from the same model run.

In [9]:
nan_per_lead = (
    weather_dataset["temperature_K"]
    .isnull()
    .sum(dim=("latitude", "longitude"))
)

grid_size = (
    weather_dataset.sizes["latitude"]
    * weather_dataset.sizes["longitude"]
)

full_missing = nan_per_lead == grid_size

partial_missing = (
    (nan_per_lead > 0)
    & (nan_per_lead < grid_size)
)

weather_dataset["lead_available"] = ~full_missing

print("Fully missing leads:", int(full_missing.sum()))
print("Partially missing leads:", int(partial_missing.sum()))
print(
    "Available leads:",
    int(weather_dataset["lead_available"].sum())
)

Fully missing leads: 6
Partially missing leads: 3
Available leads: 38954


In [10]:
temperature_filled = (
    weather_dataset["temperature_K"]
    .interpolate_na(
        dim="forecast_hour",
        method="linear",
    )
)

print(
    "Remaining missing temperature cells:",
    int(temperature_filled.isnull().sum())
)

Remaining missing temperature cells: 0


In [11]:
run = pd.Timestamp("2016-02-20")
fh = 39

left = weather_dataset["temperature_K"].sel(
    run_time_utc=run,
    forecast_hour=36,
)

right = weather_dataset["temperature_K"].sel(
    run_time_utc=run,
    forecast_hour=42,
)

filled = temperature_filled.sel(
    run_time_utc=run,
    forecast_hour=fh,
)

expected = (left + right) / 2

assert np.allclose(
    filled,
    expected,
    equal_nan=False,
)

print("Interpolation check passed.")

Interpolation check passed.


In [12]:
weather_processed = weather_dataset.copy()
weather_processed["temperature_K"] = temperature_filled

weather_processed = weather_processed.assign_coords(
    valid_time_utc=(
        weather_processed["run_time_utc"]
        + weather_processed["forecast_hour"].astype(
            "timedelta64[h]"
        )
    )
)

In [13]:
weather_processed["valid_time_utc"].isel(
    run_time_utc=0
).values

array(['2015-12-31T21:00:00.000000000', '2016-01-01T00:00:00.000000000',
       '2016-01-01T03:00:00.000000000', '2016-01-01T06:00:00.000000000',
       '2016-01-01T09:00:00.000000000', '2016-01-01T12:00:00.000000000',
       '2016-01-01T15:00:00.000000000', '2016-01-01T18:00:00.000000000',
       '2016-01-01T21:00:00.000000000', '2016-01-02T00:00:00.000000000'],
      dtype='datetime64[ns]')

## Spatial reduction to representative cities

The raw GFS data cover a regular 0.25° grid over mainland Spain. To obtain a
compact set of weather features, we select the nearest GFS grid point to eight
major Spanish cities distributed across different climatic regions.

These locations are later used as temperature features for the demand model.

In [14]:
cities_coordinates = pd.DataFrame.from_dict(
    {
        "Madrid": (40.4165, -3.7026),
        "Barcelona": (41.3888, 2.1590),
        "Valencia": (39.4767, -0.3744),
        "Sevilla": (37.3828, -5.9732),
        "Bilbao": (43.2627, -2.9253),
        "Zaragoza": (41.6561, -0.8773),
        "Málaga": (36.7194, -4.4200),
        "A Coruña": (43.3713, -8.3896),
    },
    orient="index",
    columns=["latitude", "longitude"],
)

cities_coordinates.index.name = "location"

cities_coordinates

,latitude,longitude
location,,
Madrid,40.4165,-3.7026
Barcelona,41.3888,2.1590
Valencia,39.4767,-0.3744
Sevilla,37.3828,-5.9732
Bilbao,43.2627,-2.9253
Zaragoza,41.6561,-0.8773
Málaga,36.7194,-4.4200
A Coruña,43.3713,-8.3896


In [15]:
target_latitudes = xr.DataArray(
    cities_coordinates["latitude"].to_numpy(),
    dims="location",
    coords={"location": cities_coordinates.index},
)

target_longitudes = xr.DataArray(
    cities_coordinates["longitude"].to_numpy(),
    dims="location",
    coords={"location": cities_coordinates.index},
)

In [16]:
selected_grid = weather_processed.sel(
    latitude=target_latitudes,
    longitude=target_longitudes,
    method="nearest",
)

In [17]:
selected_grid_points = pd.DataFrame(
    {
        "latitude": selected_grid["latitude"].values,
        "longitude": selected_grid["longitude"].values,
    },
    index=selected_grid["location"].values,
)

selected_grid_points.index.name = "location"

selected_grid_points

,latitude,longitude
location,,
Madrid,40.50,-3.75
Barcelona,41.50,2.25
Valencia,39.50,-0.25
Sevilla,37.50,-6.00
Bilbao,43.25,-3.00
Zaragoza,41.75,-1.00
Málaga,36.75,-4.50
A Coruña,43.25,-8.50


In [18]:
weather_by_location = selected_grid[
    ["temperature_K"]
]

## Hourly interpolation

The archived GFS forecasts are sampled every 3 hours. Since electricity demand
is modeled hourly, temperatures are linearly interpolated to forecast hours
21–48 at one-hour resolution.

In [19]:
hourly_forecast_hours = np.arange(21, 49)

temperature_hourly = (
    weather_by_location["temperature_K"]
    .interp(
        forecast_hour=hourly_forecast_hours,
        method="linear",
    )
)

weather_hourly = temperature_hourly.to_dataset(
    name="temperature_K"
)

In [20]:
print(weather_hourly.sizes)
print(
    "Missing temperatures:",
    int(weather_hourly["temperature_K"].isnull().sum()),
)

Frozen({'location': 8, 'run_time_utc': 3896, 'forecast_hour': 28})
Missing temperatures: 0


## Local D+1 forecast window

Forecast valid times are stored in UTC, but electricity demand is defined on
the Spanish local calendar. For each GFS run on day D−1, we therefore retain
only forecast hours whose valid time falls on local calendar day D in
`Europe/Madrid`.

This automatically preserves daylight-saving transitions, including 23-hour
and 25-hour local days.

In [21]:
temperature_long = (
    weather_hourly["temperature_K"]
    .to_dataframe()
    .reset_index()
)

In [22]:
temperature_long["run_time_utc"] = pd.to_datetime(
    temperature_long["run_time_utc"],
    utc=True,
)

temperature_long["valid_time_utc"] = pd.to_datetime(
    temperature_long["valid_time_utc"],
    utc=True,
)

temperature_long["valid_time_local"] = (
    temperature_long["valid_time_utc"]
    .dt.tz_convert("Europe/Madrid")
)

In [23]:
target_day = (
    temperature_long["run_time_utc"]
    .dt.tz_localize(None)
    .dt.normalize()
    + pd.Timedelta(days=1)
)

local_day = (
    temperature_long["valid_time_local"]
    .dt.tz_localize(None)
    .dt.normalize()
)

temperature_d1_long = temperature_long.loc[
    local_day == target_day
].copy()

In [24]:
temperature_d1 = (
    temperature_d1_long
    .pivot(
        index=[
            "run_time_utc",
            "forecast_hour",
            "valid_time_utc",
            "valid_time_local",
        ],
        columns="location",
        values="temperature_K",
    )
    .sort_index()
)

In [25]:
hours_per_run = (
    temperature_d1
    .groupby(level="run_time_utc")
    .size()
)

hours_per_run.value_counts().sort_index()

23      11
24    3875
25      10
Name: count, dtype: int64

In [26]:
print("Rows:", len(temperature_d1))
print(
    "Duplicated UTC instants:",
    temperature_d1.index
    .get_level_values("valid_time_utc")
    .duplicated()
    .sum(),
)
print(
    "Missing temperatures:",
    int(temperature_d1.isna().sum().sum()),
)

Rows: 93503
Duplicated UTC instants: 0
Missing temperatures: 0


## Final hourly weather features

The D+1 weather forecasts are converted back to a unique UTC time index for
merging with electricity-demand data. Temperatures are converted from Kelvin
to Celsius and stored as one feature per representative city.

In [27]:
weather_features = (
    temperature_d1
    .reset_index()
    .drop(columns=[
        "forecast_hour",
        "valid_time_local",
    ])
    .set_index("valid_time_utc")
    .sort_index()
)

weather_features.index.name = "datetime_utc"
weather_features.columns.name = None

In [28]:
weather_features = weather_features.rename(
    columns={
        "Madrid": "temp_madrid_C",
        "Barcelona": "temp_barcelona_C",
        "Valencia": "temp_valencia_C",
        "Sevilla": "temp_sevilla_C",
        "Bilbao": "temp_bilbao_C",
        "Zaragoza": "temp_zaragoza_C",
        "Málaga": "temp_malaga_C",
        "A Coruña": "temp_a_coruna_C",
    }
)

temp_cols = [
    "temp_madrid_C",
    "temp_barcelona_C",
    "temp_valencia_C",
    "temp_sevilla_C",
    "temp_bilbao_C",
    "temp_zaragoza_C",
    "temp_malaga_C",
    "temp_a_coruna_C",
]

weather_features[temp_cols] = (
    weather_features[temp_cols] - 273.15
)

In [29]:
weather_features = weather_features[
    ["run_time_utc"] + temp_cols
]

In [30]:
assert weather_features.shape == (93503, 9)

assert weather_features.index.is_unique
assert weather_features.index.is_monotonic_increasing

assert weather_features.isna().sum().sum() == 0

assert np.isfinite(
    weather_features[temp_cols].to_numpy()
).all()

assert (
    weather_features[temp_cols].abs() < 100
).all().all()

print("Shape:", weather_features.shape)
print("Start:", weather_features.index.min())
print("End:", weather_features.index.max())

Shape: (93503, 9)
Start: 2015-12-31 23:00:00+00:00
End: 2026-08-31 21:00:00+00:00
